In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import numpy as np

# Data Setup
try:
    df_local = pd.read_csv('Human Drugs Products Local.csv')
    df_imported = pd.read_csv('Imported human drug products.csv')
except FileNotFoundError as e:
    print(f"CRITICAL ERROR: Please verify the file names and ensure both files are uploaded. Error: {e}")
    exit()

manufacturer_col_local = [col for col in df_local.columns if 'Manufacture' in col and 'Country' not in col][0]
df_local.rename(columns={manufacturer_col_local: 'Manufacture Name'}, inplace=True)
df_local['Source_Type'] = 'Local Production'
df_imported['Source_Type'] = 'Imported Drugs'
columns_to_drop = ['Unnamed: 10', 'Unnamed: 11']
df_local.drop(columns=columns_to_drop, inplace=True, errors='ignore')
df_imported.drop(columns=columns_to_drop, inplace=True, errors='ignore')

df_combined = pd.concat([df_local, df_imported], ignore_index=True)
df_local_only = df_combined[df_combined['Source_Type'] == 'Local Production'].copy()

print("Advanced Model Preparation: Predicting Technological Readiness ")

advanced_companies = df_local_only[
    (df_local_only['DrugType'] == 'NCE') | (df_local_only['DrugType'] == 'Biological')
]['Manufacture Name'].unique()

df_local_only['Is_Advanced_Producer'] = df_local_only['Manufacture Name'].apply(
    lambda x: 1 if x in advanced_companies else 0
)

features = ['SizeUnit', 'LegalStatus']
target = 'Is_Advanced_Producer'

X = df_local_only[features]
X_encoded = pd.get_dummies(X, columns=features, drop_first=True)
y = df_local_only[target]

print(f"Total Local Products Analyzed: {df_local_only.shape[0]}")
print(f"Number of Advanced Producers (Target=1): {y.sum()} / (Target=0): {df_local_only.shape[0] - y.sum()}")
print("-" * 50)


X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.3, random_state=42, stratify=y)

print("Decision Tree Model: Performance (Using Class Weights) ")

dt_capacity_model_weighted = DecisionTreeClassifier(
    max_depth=5,
    random_state=42,
    class_weight='balanced'
)
dt_capacity_model_weighted.fit(X_train, y_train)
dt_capacity_predictions_weighted = dt_capacity_model_weighted.predict(X_test)

print(f"\n Model Performance Summary ")
print(f"Accuracy (Overall): {accuracy_score(y_test, dt_capacity_predictions_weighted):.4f}")
print("Classification Report (Detailed Metrics)")
print(classification_report(
    y_test,
    dt_capacity_predictions_weighted,
    target_names=['Generic Focus (0)', 'Advanced Capacity (1)']
))
print("-" * 50)


print("\nTop Features Driving Advanced Production Capacity  ")
importance_df_weighted = pd.DataFrame({
    'Feature': X_encoded.columns,
    'Importance': dt_capacity_model_weighted.feature_importances_
}).sort_values(by='Importance', ascending=False).head(5)

print("Top 5 Product Characteristics Indicating Advanced Manufacturer Capacity:")
print(importance_df_weighted.to_markdown(index=False, numalign="left", stralign="left"))
print("-" * 50)

Advanced Model Preparation: Predicting Technological Readiness 
Total Local Products Analyzed: 2521
Number of Advanced Producers (Target=1): 1649 / (Target=0): 872
--------------------------------------------------
Decision Tree Model: Performance (Using Class Weights) 

 Model Performance Summary 
Accuracy (Overall): 0.5376
Classification Report (Detailed Metrics)
                       precision    recall  f1-score   support

    Generic Focus (0)       0.40      0.66      0.50       262
Advanced Capacity (1)       0.72      0.47      0.57       495

             accuracy                           0.54       757
            macro avg       0.56      0.57      0.53       757
         weighted avg       0.61      0.54      0.55       757

--------------------------------------------------

Top Features Driving Advanced Production Capacity  
Top 5 Product Characteristics Indicating Advanced Manufacturer Capacity:
| Feature                  | Importance   |
|:-------------------------|:-